In [1]:
# data

from data.get import get_data
from data.splits import load_splits

In [2]:
import torch
import torch.nn as nn

from spdnet.spd import SPDTransform
from eegspdnet.nn import ChSpecConv, SCMPool, AltLogEig, AltReEig
from eegspdnet.optim import AltStiefelMetaOptimizer


In [3]:
defaults = {
    'channels_to_pick': sorted(['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'C4', 'CP5', 'CP1', 'CP2', 'CP6', 'FC3', 'FCz',
                                'FC4', 'C5', 'C1', 'C2', 'C6', 'CP3', 'CPz', 'CP4', 'FFC5h', 'FFC3h', 'FFC4h',
                                'FFC6h', 'FCC5h', 'FCC3h', 'FCC4h', 'FCC6h', 'CCP5h', 'CCP3h', 'CCP4h', 'CCP6h',
                                'CPP5h', 'CPP3h', 'CPP4h', 'CPP6h', 'FFC1h', 'FFC2h', 'FCC1h', 'FCC2h', 'CCP1h',
                                'CCP2h', 'CPP1h', 'CPP2h']),
    'low_cut_hz': 4,
    'hi_cut_hz': 124,
    's_freq': 250,
    'max_abs_val': 800,
    'trial_stop_offset_samples': 0,
    'trial_start_offset_samples': 125,
    'n_jobs': 1,
    'validation_split_point': 0.8,
}

params = {
    'batch_size': 256,
    'shuffle': False,
    'drop_last': False
}

In [4]:
mode='valid'
sub=1
name='Schirrmeister2017'

trainset, testset = load_splits(*get_data(sub, name=name, **defaults), mode=mode, **params)

Creating RawArray with float64 data, n_channels=128, n_times=1225545
    Range : 0 ... 1225544 =      0.000 ...  2451.088 secs
Ready.
Creating RawArray with float64 data, n_channels=128, n_times=616535
    Range : 0 ... 616534 =      0.000 ...  1233.068 secs
Ready.
320 events found
Event IDs: [1 2 3 4]
160 events found
Event IDs: [1 2 3 4]
Used Annotations descriptions: ['feet', 'left_hand', 'rest', 'right_hand']
Adding metadata with 4 columns
Replacing existing metadata with 4 columns
320 matching events found
No baseline correction applied
0 projection items activated
Loading data for 320 events and 875 original time points ...
0 bad epochs dropped
Loading data for 320 events and 875 original time points ...
Used Annotations descriptions: ['feet', 'left_hand', 'rest', 'right_hand']
Adding metadata with 4 columns
Replacing existing metadata with 4 columns
160 matching events found
No baseline correction applied
0 projection items activated
Loading data for 160 events and 875 original 

In [5]:
n_bands = 4
n_classes = 4
n_ch = len(defaults['channels_to_pick'])

In [6]:
class EEGSPDNet(nn.Module):
    
    def __init__(self, n_ch, n_bands, n_classes):
        super(__class__, self).__init__()
    
        self.conv = ChSpecConv(n_ch=n_ch, n_filters=n_bands, n_time_kernel=25)
        self.scm = SCMPool(n_bands, add_d=False)
        
        bimap_sizes = [(n_bands * n_ch) // 2**i for i in range(4)]
        self.bimap1 = SPDTransform(bimap_sizes[0], bimap_sizes[1])
        self.reeig1 = AltReEig()
        self.bimap2 = SPDTransform(bimap_sizes[1], bimap_sizes[2])
        self.reeig2 = AltReEig()
        self.bimap3 = SPDTransform(bimap_sizes[2], bimap_sizes[3])
        self.reeig3 = AltReEig()
        self.log = AltLogEig(bimap_sizes[-1])
        
        vectorised_size = int(bimap_sizes[-1] * (bimap_sizes[-1] + 1) * 0.5)
        
        self.linear = nn.Linear(vectorised_size, n_classes)
        
    def forward(self, x):
        
        x = self.conv(x)
        x = self.scm(x)
        x = self.bimap1(x)
        x = self.reeig1(x)
        x = self.bimap2(x)
        x = self.reeig2(x)
        x = self.bimap3(x)
        x = self.reeig3(x)
        x = self.log(x)
        x = self.linear(x.flatten(start_dim=1))
        return x
    
model = EEGSPDNet(n_ch=n_ch, n_bands=n_bands, n_classes=n_classes)
_optim = torch.optim.Adam(model.parameters(), lr=1e-3)
optim = AltStiefelMetaOptimizer(_optim)
criterion = nn.CrossEntropyLoss()

In [7]:
n_epochs = 50

# train

for epoch in range(n_epochs):
    
    model.train()
    
    train_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, batch in enumerate(trainset):
        data = batch['data']
        labels = batch['label'].squeeze().long()
        
        optim.zero_grad()
        outputs = model(data)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()
        
        train_loss += loss.data.item()
        _, preds = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += preds.eq(labels.data).cpu().sum().data.item()
        
    final_loss = train_loss / (batch_idx + 1)
    acc = 100 * (correct / total)
    
    print(f'{epoch}: {final_loss:.2f}, {acc:.2f}%')

0: 1.44, 25.00%
1: 1.40, 25.00%
2: 1.37, 41.02%
3: 1.36, 35.16%
4: 1.34, 42.19%
5: 1.33, 30.86%
6: 1.32, 29.69%
7: 1.30, 34.77%
8: 1.29, 44.53%
9: 1.27, 57.03%
10: 1.26, 66.02%
11: 1.24, 70.70%
12: 1.23, 65.62%
13: 1.22, 63.67%
14: 1.21, 63.28%
15: 1.20, 63.67%
16: 1.18, 66.41%
17: 1.17, 67.97%
18: 1.16, 69.92%
19: 1.14, 71.88%
20: 1.13, 72.66%
21: 1.12, 71.88%
22: 1.11, 71.09%
23: 1.09, 71.88%
24: 1.08, 73.83%
25: 1.07, 73.44%
26: 1.05, 72.66%
27: 1.04, 73.44%
28: 1.03, 73.44%
29: 1.02, 72.66%
30: 1.00, 74.61%
31: 0.99, 72.66%
32: 0.98, 73.44%
33: 0.96, 73.44%
34: 0.95, 75.39%
35: 0.94, 74.61%
36: 0.93, 75.39%
37: 0.91, 75.00%
38: 0.90, 74.61%
39: 0.89, 73.83%
40: 0.88, 74.61%
41: 0.86, 74.61%
42: 0.85, 75.00%
43: 0.84, 77.34%
44: 0.83, 76.56%
45: 0.82, 78.52%
46: 0.81, 78.12%
47: 0.80, 77.34%
48: 0.79, 77.34%
49: 0.78, 76.95%


In [9]:
# test

model.eval()

test_loss = 0
correct = 0
total = 0

for batch_idx, batch in enumerate(testset):
    data = batch['data']
    labels = batch['label'].squeeze().long()
    
    outputs = model(data)
    
    loss = criterion(outputs, labels)
    
    test_loss += loss.data.item()
    _, preds = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += preds.eq(labels.data).cpu().sum().data.item()
    
final_loss = train_loss / (batch_idx + 1)
acc = 100 * (correct / total)

print(final_loss, acc)

0.7756412625312805 73.4375
